In [1]:
#conect with drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "xx"
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"] = "xx"

In [7]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/fetch_src.sh  # Windows users
!bash /content/drive/MyDrive/apai/fetch_src.sh

 fetch_src.sh — Download /src from GitHub

Checking environment...
GITHUB_TOKEN set
Fetching file list from GitHub API...
Found 19 file(s) under /src/
src/cl_strategies/ewc.py -> /content/src/cl_strategies/ewc.py
src/cl_strategies/naive.py -> /content/src/cl_strategies/naive.py
src/cl_strategies/reharsal.py -> /content/src/cl_strategies/reharsal.py
src/config/config.py -> /content/src/config/config.py
src/data/dataset.py -> /content/src/data/dataset.py
src/data/preprocessing.py -> /content/src/data/preprocessing.py
src/data/study_dataset.py -> /content/src/data/study_dataset.py
src/fine_tune/trainer.py -> /content/src/fine_tune/trainer.py
src/fine_tune/visualizer.py -> /content/src/fine_tune/visualizer.py
src/imports.py -> /content/src/imports.py
src/meta_learning/buffers.py -> /content/src/meta_learning/buffers.py
src/meta_learning/data.py -> /content/src/meta_learning/data.py
src/meta_learning/embeddings.py -> /content/src/meta_learning/embeddings.py
src/meta_learning/models.py -> /c

In [4]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/setup_colab.sh    # Windows users
!bash /content/drive/MyDrive/apai/setup_colab.sh


UCF101 Project — Colab Setup Script

Checking environment...
KAGGLE_USERNAME set
KAGGLE_KEY set
config.py found

Installing kagglehub...
Done.

100% 6.53G/6.53G [01:12<00:00, 96.9MB/s]
Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4
/content/drive/MyDrive/apai/setup_colab.sh: error reading input file: Transport endpoint is not connected


In [8]:
# imports
%run /content/src/imports.py

Using device: cuda


In [9]:
# preprocessing roots
TASK0_ROOT           = "./UCF101/processed_task0"
EXEMPLAR_ROOT        = "./UCF101/processed_exemplars"
SUBSET1_ROOT         = "./UCF101/processed_subset1"
LIMITED_SUBSET1_ROOT = "./UCF101/processed_limited_subset1"
SUBSET2_ROOT         = "./UCF101/processed_subset2"
LIMITED_SUBSET2_ROOT = "./UCF101/processed_limited_subset2"

# embedding roots
TASK0_EMB_ROOT    = "./UCF101/emb_task0"
EXEMPLAR_EMB_ROOT = "./UCF101/emb_exemplars"
SUBSET1_EMB_ROOT  = "./UCF101/emb_subset1"
LIM_SUB1_EMB_ROOT = "./UCF101/emb_limited_subset1"
SUBSET2_EMB_ROOT  = "./UCF101/emb_subset2"
LIM_SUB2_EMB_ROOT = "./UCF101/emb_limited_subset2"

ALL_EMB_ROOTS = [
    TASK0_EMB_ROOT, EXEMPLAR_EMB_ROOT,
    SUBSET1_EMB_ROOT, LIM_SUB1_EMB_ROOT,
    SUBSET2_EMB_ROOT, LIM_SUB2_EMB_ROOT,
]


In [10]:
# data budget
LIMIT      = 5    # few-shot -> 5 training samples
VAL_CAP    = 20   # validation samples
BATCH_SIZE = 8

# training
NEW_REPEAT = 10   # up-sample new-task data × this factor per epoch

# Optuna
N_TRIALS        = 15   # trials
SEARCH_EPOCHS   = 3    # epochs
SEARCH_EPISODES = 80   # reptile episodes

# reptile
REP_N_WAY     = 3    # 3 new classes per episode
REP_K_SUPPORT = 3    # support shots per class
REP_K_QUERY   = 2    # query shots per class
REP_LAMBDA_KD = 0.3  # KD loss weight

T1_INNER_STEPS = 5; T1_EPISODES = 200; T1_EPOCHS = 5; T1_NEW_BIAS = 3
T2_INNER_STEPS = 5; T2_EPISODES = 300; T2_EPOCHS = 5; T2_NEW_BIAS = 4

# class index 
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx     = {cls: i for i, cls in enumerate(ALL_CLASSES_LIST)}

N0      = len(cfg.SELECTED_CLASSES)   # old classes
N1      = len(cfg.TASK_1)             # Task-1 new classes
N2      = len(cfg.TASK_2)             # Task-2 new classes
N_TOTAL = N0 + N1 + N2               # full head size

TASK1_CLASS_IDS = list(range(N0, N0 + N1))           # indices [10..12]
TASK2_CLASS_IDS = list(range(N0 + N1, N_TOTAL))      # indices [13..15]

### **Preprocess**

In [11]:
# preprocess just the val,test on the old classes
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=TASK0_ROOT, max_samples=VAL_CAP, splits=["val"])
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=TASK0_ROOT, splits=["test"])          

# old classes trian limited for reply buffer
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=EXEMPLAR_ROOT, max_samples=LIMIT, splits=["train"])

# task 1 few-shot
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=LIMITED_SUBSET1_ROOT, max_samples=LIMIT, splits=["train"])
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=SUBSET1_ROOT, max_samples=VAL_CAP, splits=["val"])
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=SUBSET1_ROOT, splits=["test"])

# task 2 few-shot
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=LIMITED_SUBSET2_ROOT, max_samples=LIMIT, splits=["train"])
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=SUBSET2_ROOT, max_samples=VAL_CAP, splits=["val"])
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=SUBSET2_ROOT, splits=["test"])


--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_task0
Classes: 10 | Limit: 20

Processing split: val



Done! Dataset ready at: ./UCF101/processed_task0

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_task0
Classes: 10 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_task0

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_exemplars
Classes: 10 | Limit: 5

Processing split: train



Done! Dataset ready at: ./UCF101/processed_exemplars

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_limited_subset1
Classes: 3 | Limit: 5

Processing split: train



Done! Dataset ready at: ./UCF101/processed_limited_subset1

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset1
Classes: 3 | Limit: 20

Processing split: val



Done! Dataset ready at: ./UCF101/processed_subset1

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset1
Classes: 3 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_subset1

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_limited_subset2
Classes: 3 | Limit: 5

Processing split: train



Done! Dataset ready at: ./UCF101/processed_limited_subset2

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset2
Classes: 3 | Limit: 20

Processing split: val



Done! Dataset ready at: ./UCF101/processed_subset2

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset2
Classes: 3 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_subset2


### **Upload Model**
- TODO: make the validation on the new tasks befor exctracting

In [12]:
model = ResNet50LSTM(num_classes=N0).to(device)
model.load_state_dict(torch.load(cfg.RESNET50_PATH, map_location=device))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 171MB/s]


<All keys matched successfully>

In [13]:

model_embeddings = ResNet50Trunk(model).to(device)
model_embeddings.eval() # evaluation
del model  # keep only the backbone

### **Exctract embeddings**

- TODO: Visualise

In [14]:
extraction_plan = [
    (TASK0_ROOT,           TASK0_EMB_ROOT,    "Task 0        -> val&test"),
    (EXEMPLAR_ROOT,        EXEMPLAR_EMB_ROOT, f"Exemplars     -> {LIMIT}/cls train  <- replay buffer"),
    (LIMITED_SUBSET1_ROOT, LIM_SUB1_EMB_ROOT, f"Task 1 train  -> {LIMIT}/cls (few-shot)"),
    (SUBSET1_ROOT,         SUBSET1_EMB_ROOT,  "Task 1        -> val & test"),
    (LIMITED_SUBSET2_ROOT, LIM_SUB2_EMB_ROOT, f"Task 2 train  -> {LIMIT}/cls (few-shot)"),
    (SUBSET2_ROOT,         SUBSET2_EMB_ROOT,  "Task 2        -> val & test"),
]

# delete existing dirs
for d in ALL_EMB_ROOTS:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"  Deleted stale dir: {d}")

for clip_root, emb_root, label in extraction_plan:
    print(f"  -> {label}")
    extract_embeddings(clip_root, emb_root, model_embeddings, device)
    torch.cuda.empty_cache()

del model_embeddings
torch.cuda.empty_cache()

  -> Task 0        -> val&test


  -> Exemplars     -> 5/cls train  <- replay buffer


  -> Task 1 train  -> 5/cls (few-shot)


  -> Task 1        -> val & test


  -> Task 2 train  -> 5/cls (few-shot)


  -> Task 2        -> val & test


### **Loaders**

In [15]:
# Task 0: val & test
task0_val_ds  = UCF101Embeddings(f"{TASK0_EMB_ROOT}/val",  class_to_idx)
task0_test_ds = UCF101Embeddings(f"{TASK0_EMB_ROOT}/test", class_to_idx)

# exemplars limit classes from T0 train
exemplar_train_ds = UCF101Embeddings(f"{EXEMPLAR_EMB_ROOT}/train", class_to_idx)

# Task 1: few-shot 
task1_train_ds = UCF101Embeddings(f"{LIM_SUB1_EMB_ROOT}/train", class_to_idx)
task1_val_ds   = UCF101Embeddings(f"{SUBSET1_EMB_ROOT}/val",    class_to_idx)
task1_test_ds  = UCF101Embeddings(f"{SUBSET1_EMB_ROOT}/test",   class_to_idx)

# Task 2: few-shot
task2_train_ds = UCF101Embeddings(f"{LIM_SUB2_EMB_ROOT}/train", class_to_idx)
task2_val_ds   = UCF101Embeddings(f"{SUBSET2_EMB_ROOT}/val",    class_to_idx)
task2_test_ds  = UCF101Embeddings(f"{SUBSET2_EMB_ROOT}/test",   class_to_idx)

# DataLoaders
task0_val_loader      = _dl(task0_val_ds)
task0_test_loader     = _dl(task0_test_ds)
exemplar_train_loader = _dl(exemplar_train_ds)   # shuffle=False

task1_train_loader = _dl(task1_train_ds, shuffle=True)   # shuffled: new task training
task1_val_loader   = _dl(task1_val_ds)
task1_test_loader  = _dl(task1_test_ds)

task2_train_loader = _dl(task2_train_ds, shuffle=True)
task2_val_loader   = _dl(task2_val_ds)
task2_test_loader  = _dl(task2_test_ds)

# combined loaders
# val
combined_val_t1_loader  = _dl(ConcatDataset([task0_val_ds, task1_val_ds]), shuffle=True)
combined_val_t2_loader  = _dl(ConcatDataset([task0_val_ds, task1_val_ds, task2_val_ds]),
                               shuffle=True)
# test
combined_test_t1_loader = _dl(ConcatDataset([task0_test_ds, task1_test_ds]))
combined_test_t2_loader = _dl(ConcatDataset([task0_test_ds, task1_test_ds, task2_test_ds]))

# combined exemplars
combined_exemplar_t2_ds     = ConcatDataset([exemplar_train_ds, task1_train_ds])
combined_exemplar_t2_loader = _dl(combined_exemplar_t2_ds)

## **Rehersial**

In [16]:
# search best lr + lambda_distill for Rehearsal without KD
best_reh_params_t1 = optuna_search_rehearsal(
    task_tag="T1-noKD",
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    exemplar_train_loader=exemplar_train_loader,
    extra_replay_loader=None,          # no extra replay for T1
    teacher=None,                      # no KD
    cfg=cfg, device=device, lstm_hidden=256,
    num_classes=N_TOTAL,
    num_old_classes=N0,
    limit=LIMIT,
    kd=False,
    n_trials=N_TRIALS,
    search_epochs=SEARCH_EPOCHS,
    new_repeat=NEW_REPEAT
)
print("Best Rehearsal (no KD) params for T1:", best_reh_params_t1)

[I 2026-05-10 16:29:48,762] A new study created in memory with name: no-name-1601ff31-09a8-4b63-95fc-4b5c803ab08c



OPTUNA REHEARSAL — T1-noKD | 15 trials × 3 epochs


  0%|          | 0/15 [00:00<?, ?it/s]

[Rehearsal no-KD] Epoch 1/3 | Loss: 13.5612 | ValAcc: 0.8186
[Rehearsal no-KD] Epoch 2/3 | Loss: 0.4696 | ValAcc: 0.8279
[Rehearsal no-KD] Epoch 3/3 | Loss: 0.1275 | ValAcc: 0.8372


/content/src/meta_learning/optuna_search.py:177: UserWarning: The reported value is ignored because this `step` 2 is already reported.
  trial.report(val_acc, search_epochs - 1)


[I 2026-05-10 16:29:54,255] Trial 0 finished with value: 0.8372093023255814 and parameters: {'lambda_distill': 0.4, 'lstm_lr': 0.0007969454818643932, 'fc_lr': 0.0029106359131330704}. Best is trial 0 with value: 0.8372093023255814.
[Rehearsal no-KD] Epoch 1/3 | Loss: 45.8566 | ValAcc: 0.3814
[Rehearsal no-KD] Epoch 2/3 | Loss: 31.2483 | ValAcc: 0.5488
[Rehearsal no-KD] Epoch 3/3 | Loss: 19.1914 | ValAcc: 0.6093
[I 2026-05-10 16:30:00,501] Trial 1 finished with value: 0.6093023255813953 and parameters: {'lambda_distill': 0.6000000000000001, 'lstm_lr': 2.0513382630874486e-05, 'fc_lr': 0.00020511104188433984}. Best is trial 0 with value: 0.8372093023255814.
[Rehearsal no-KD] Epoch 1/3 | Loss: 16.9038 | ValAcc: 0.8558
[Rehearsal no-KD] Epoch 2/3 | Loss: 0.9860 | ValAcc: 0.8419
[Rehearsal no-KD] Epoch 3/3 | Loss: 0.2757 | ValAcc: 0.8326
[I 2026-05-10 16:30:05,354] Trial 2 finished with value: 0.8325581395348837 and parameters: {'lambda_distill': 0.1, 'lstm_lr': 0.0005399484409787432, 'fc_lr'

In [19]:
# build model + optimizer with best params
m_reh = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)
optimizer_reh = torch.optim.Adam(
    [
        {"params": m_reh.lstm.parameters(), "lr": best_reh_params_t1["lstm_lr"]},
        {"params": m_reh.fc.parameters(),   "lr": best_reh_params_t1["fc_lr"]},
    ],
    weight_decay=1e-4
)

# build replay buffer with T0 exemplars
replay_buf_t1 = fresh_replay_buffer(exemplar_train_loader=exemplar_train_loader, limit=LIMIT)

# final training run with best params
train_rehearsal(
    model=m_reh,
    teacher=None,
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    replay_buffer=replay_buf_t1,
    optimizer=optimizer_reh,
    device=device,
    num_old_classes=N0,
    lambda_distill=best_reh_params_t1["lambda_distill"],
    epochs=T1_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=False
)

[Rehearsal no-KD] Epoch 1/5 | Loss: 10.9979 | ValAcc: 0.8558
[Rehearsal no-KD] Epoch 2/5 | Loss: 0.2874 | ValAcc: 0.8465
[Rehearsal no-KD] Epoch 3/5 | Loss: 0.0742 | ValAcc: 0.8558
[Rehearsal no-KD] Epoch 4/5 | Loss: 0.0961 | ValAcc: 0.8233
[Rehearsal no-KD] Epoch 5/5 | Loss: 0.0523 | ValAcc: 0.8465


In [20]:
# recording meta-learning results
results_t1, results_t2 = {}, {}

def record(name, model, results_dict, old_loader, new_loader, all_loader):
    _, old     = evaluate(model, old_loader, device)
    _, new_    = evaluate(model, new_loader, device)
    _, all_acc = evaluate(model, all_loader, device)
    results_dict[name] = {"Old": old, "New": new_, "All": all_acc}
    print(f" {name:<36} Old:{old:.4f}  New:{new_:.4f}  All:{all_acc:.4f}")

    
record(
    name="Rehearsal T1 (no KD, best)", model=m_reh, results_dict=results_t1,
    old_loader=task0_test_loader, new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader
)

 Rehearsal T1 (no KD, best)           Old:0.9415  New:0.6226  All:0.8661


In [21]:
# frozen teacher used for KD loss — untrained fresh model acts as a fixed reference
teacher_base = make_teacher(fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256))

# search best lr + lambda_distill for Rehearsal with KD
best_reh_kd_params_t1 = optuna_search_rehearsal(
    task_tag="T1-KD",
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    exemplar_train_loader=exemplar_train_loader,
    extra_replay_loader=None,
    teacher=teacher_base,
    cfg=cfg, device=device, lstm_hidden=256,
    num_classes=N_TOTAL,
    num_old_classes=N0,
    limit=LIMIT,
    kd=True,
    n_trials=N_TRIALS,
    search_epochs=SEARCH_EPOCHS,
    new_repeat=NEW_REPEAT
)
print("Best Rehearsal (KD) params for T1:", best_reh_kd_params_t1)

[I 2026-05-10 16:33:06,833] A new study created in memory with name: no-name-f486f1df-c086-4e9b-9ae6-31fe3c067031



OPTUNA REHEARSAL — T1-KD | 15 trials × 3 epochs


  0%|          | 0/15 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1141: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)
  result = _VF.lstm(


[Rehearsal KD] Epoch 1/3 | Loss: 16.7292 | ValAcc: 0.8279
[Rehearsal KD] Epoch 2/3 | Loss: 5.1983 | ValAcc: 0.8279
[Rehearsal KD] Epoch 3/3 | Loss: 3.7720 | ValAcc: 0.8233
[I 2026-05-10 16:33:13,292] Trial 0 finished with value: 0.8232558139534883 and parameters: {'lambda_distill': 0.4, 'lstm_lr': 0.0007969454818643932, 'fc_lr': 0.0029106359131330704}. Best is trial 0 with value: 0.8232558139534883.
[Rehearsal KD] Epoch 1/3 | Loss: 49.2833 | ValAcc: 0.2419
[Rehearsal KD] Epoch 2/3 | Loss: 33.2275 | ValAcc: 0.3674
[Rehearsal KD] Epoch 3/3 | Loss: 23.8363 | ValAcc: 0.4698
[I 2026-05-10 16:33:18,414] Trial 1 finished with value: 0.4697674418604651 and parameters: {'lambda_distill': 0.6000000000000001, 'lstm_lr': 2.0513382630874486e-05, 'fc_lr': 0.00020511104188433984}. Best is trial 0 with value: 0.8232558139534883.
[Rehearsal KD] Epoch 1/3 | Loss: 19.4841 | ValAcc: 0.7860
[Rehearsal KD] Epoch 2/3 | Loss: 2.7564 | ValAcc: 0.8419
[Rehearsal KD] Epoch 3/3 | Loss: 1.8925 | ValAcc: 0.8279
[I 

In [22]:
# build model + optimizer with best params
m_reh_kd = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)
optimizer_reh_kd = torch.optim.Adam(
    [
        {"params": m_reh_kd.lstm.parameters(), "lr": best_reh_kd_params_t1["lstm_lr"]},
        {"params": m_reh_kd.fc.parameters(),   "lr": best_reh_kd_params_t1["fc_lr"]},
    ],
    weight_decay=1e-4
)

# build replay buffer with T0 exemplars
replay_buf_t1_kd = fresh_replay_buffer(exemplar_train_loader=exemplar_train_loader, limit=LIMIT)

# final training run with best params
train_rehearsal(
    model=m_reh_kd,
    teacher=teacher_base,
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    replay_buffer=replay_buf_t1_kd,
    optimizer=optimizer_reh_kd,
    device=device,
    num_old_classes=N0,
    lambda_distill=best_reh_kd_params_t1["lambda_distill"],
    epochs=T1_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=True
)

record(
    name="Rehearsal T1 (KD, best)", model=m_reh_kd, results_dict=results_t1,
    old_loader=task0_test_loader, new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader
)

[Rehearsal KD] Epoch 1/5 | Loss: 20.4927 | ValAcc: 0.7442
[Rehearsal KD] Epoch 2/5 | Loss: 8.0272 | ValAcc: 0.8605
[Rehearsal KD] Epoch 3/5 | Loss: 5.9575 | ValAcc: 0.8512
[Rehearsal KD] Epoch 4/5 | Loss: 4.6891 | ValAcc: 0.8651
[Rehearsal KD] Epoch 5/5 | Loss: 4.3927 | ValAcc: 0.8651
 Rehearsal T1 (KD, best)              Old:0.9415  New:0.7547  All:0.8973


## **REPTILE**

In [23]:
# search inner_lr, epsilon, lstm_lr, fc_lr for Reptile without KD
t1_nokd_params = optuna_search_reptile(
    task_tag="T1-noKD",
    val_loader=combined_val_t1_loader,
    new_loaders_for_buffer=[task1_train_loader],
    new_class_ids=set(TASK1_CLASS_IDS),
    search_new_weight=False,
    cfg=cfg, device=device, lstm_hidden=256,
    exemplar_train_loader=exemplar_train_loader,
    num_old_classes=N0,
    kd_flag=False, teacher=None,
    new_class_bias=T1_NEW_BIAS,
    n_trials=N_TRIALS, search_episodes=SEARCH_EPISODES, search_epochs=SEARCH_EPOCHS,
    rep_n_way=REP_N_WAY, rep_k_support=REP_K_SUPPORT, rep_k_query=REP_K_QUERY,
    rep_lambda_kd=0.0
)
print("Best Reptile (no KD) params for T1:", t1_nokd_params)


OPTUNA REPTILE — T1-noKD (no-KD) | 15 trials | episodes=80 × epochs=3


[I 2026-05-10 16:34:33,929] A new study created in memory with name: no-name-affb690a-9737-4182-9494-73f04c5df1c4


  0%|          | 0/15 [00:00<?, ?it/s]

[search-T1-noKD-t0] Epoch 1/3 | ValAcc: 0.8512 | QryLoss: 0.0297
[search-T1-noKD-t0] Epoch 2/3 | ValAcc: 0.8558 | QryLoss: 0.0306
[search-T1-noKD-t0] Epoch 3/3 | ValAcc: 0.8465 | QryLoss: 0.0095
  [search-T1-noKD-t0] Best val acc : 0.8558
[I 2026-05-10 16:34:38,967] Trial 0 finished with value: 0.8558139534883721 and parameters: {'inner_lr': 0.0030710573677773722, 'epsilon': 0.5, 'lstm_lr': 0.000291063591313307, 'fc_lr': 0.0015751320499779737}. Best is trial 0 with value: 0.8558139534883721.
[search-T1-noKD-t1] Epoch 1/3 | ValAcc: 0.8326 | QryLoss: 0.7525
[search-T1-noKD-t1] Epoch 2/3 | ValAcc: 0.8465 | QryLoss: 0.0885
[search-T1-noKD-t1] Epoch 3/3 | ValAcc: 0.8558 | QryLoss: 0.1034
  [search-T1-noKD-t1] Best val acc : 0.8558
[I 2026-05-10 16:34:44,369] Trial 1 finished with value: 0.8558139534883721 and parameters: {'inner_lr': 0.0015958237752949746, 'epsilon': 0.15000000000000002, 'lstm_lr': 1.3066739238053272e-05, 'fc_lr': 0.005399484409787433}. Best is trial 0 with value: 0.8558139

In [24]:

# extract all best params including the new lr params
T1_INNER_LR = t1_nokd_params["inner_lr"]
T1_EPSILON  = t1_nokd_params["epsilon"]
T1_LSTM_LR  = t1_nokd_params["lstm_lr"]
T1_FC_LR    = t1_nokd_params["fc_lr"]

# build episode buffer: T0 exemplars (old) + T1 train (new)
episode_buffer_t1 = EpisodeBuffer()
episode_buffer_t1.add_from_loader(exemplar_train_loader, mark_new=False)
episode_buffer_t1.add_from_loader(task1_train_loader,    mark_new=True)
n_way_eff_t1 = min(REP_N_WAY, len(episode_buffer_t1.available_classes()))

m_rep_t1 = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)

# final training run with best params
m_rep_t1, _ = train_reptile(
    model=m_rep_t1,
    episode_buffer=episode_buffer_t1,
    val_loader=combined_val_t1_loader,
    device=device,
    teacher=None,
    num_old_classes=N0,
    n_way=n_way_eff_t1,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=T1_INNER_LR,
    lstm_lr=T1_LSTM_LR,
    fc_lr=T1_FC_LR,
    inner_steps=T1_INNER_STEPS,
    epsilon=T1_EPSILON,
    episodes=T1_EPISODES,
    epochs=T1_EPOCHS,
    lambda_kd=0.0,
    new_class_bias=T1_NEW_BIAS,
    new_class_ids=set(TASK1_CLASS_IDS),
    new_weight=1.0,
    kd=False,
    tag="Reptile-T1-noKD-final",
    trial=None
)

record(
    name="Reptile T1 (no KD, best)", model=m_rep_t1, results_dict=results_t1,
    old_loader=task0_test_loader, new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader
)

[Reptile-T1-noKD-final] Epoch 1/5 | ValAcc: 0.8651 | QryLoss: 0.0043
[Reptile-T1-noKD-final] Epoch 2/5 | ValAcc: 0.8605 | QryLoss: 0.0018
[Reptile-T1-noKD-final] Epoch 3/5 | ValAcc: 0.8605 | QryLoss: 0.0010
[Reptile-T1-noKD-final] Epoch 4/5 | ValAcc: 0.8605 | QryLoss: 0.0010
[Reptile-T1-noKD-final] Epoch 5/5 | ValAcc: 0.8605 | QryLoss: 0.0012
  [Reptile-T1-noKD-final] Best val acc : 0.8651
 Reptile T1 (no KD, best)             Old:0.9591  New:0.7170  All:0.9018


In [25]:
# frozen teacher for Reptile KD loss
t1_teacher = make_teacher(fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256))

# search inner_lr, epsilon, lstm_lr, fc_lr for Reptile with KD
t1_kd_params = optuna_search_reptile(
    task_tag="T1-KD",
    val_loader=combined_val_t1_loader,
    new_loaders_for_buffer=[task1_train_loader],
    new_class_ids=set(TASK1_CLASS_IDS),
    search_new_weight=False,
    cfg=cfg, device=device, lstm_hidden=256,
    exemplar_train_loader=exemplar_train_loader,
    num_old_classes=N0,
    kd_flag=True, teacher=t1_teacher,
    new_class_bias=T1_NEW_BIAS,
    n_trials=N_TRIALS, search_episodes=SEARCH_EPISODES, search_epochs=SEARCH_EPOCHS,
    rep_n_way=REP_N_WAY, rep_k_support=REP_K_SUPPORT, rep_k_query=REP_K_QUERY,
    rep_lambda_kd=REP_LAMBDA_KD
)
print("Best Reptile (KD) params for T1:", t1_kd_params)


OPTUNA REPTILE — T1-KD (KD) | 15 trials | episodes=80 × epochs=3


[I 2026-05-10 16:36:08,095] A new study created in memory with name: no-name-a9ab4d88-8686-447e-9a76-9fc7f290a678


  0%|          | 0/15 [00:00<?, ?it/s]

[search-T1-KD-t0] Epoch 1/3 | ValAcc: 0.8326 | QryLoss: 0.0599
[search-T1-KD-t0] Epoch 2/3 | ValAcc: 0.8419 | QryLoss: 0.0521
[search-T1-KD-t0] Epoch 3/3 | ValAcc: 0.8419 | QryLoss: 0.0331
  [search-T1-KD-t0] Best val acc : 0.8419
[I 2026-05-10 16:36:14,923] Trial 0 finished with value: 0.8418604651162791 and parameters: {'inner_lr': 0.0030710573677773722, 'epsilon': 0.5, 'lstm_lr': 0.000291063591313307, 'fc_lr': 0.0015751320499779737}. Best is trial 0 with value: 0.8418604651162791.
[search-T1-KD-t1] Epoch 1/3 | ValAcc: 0.8093 | QryLoss: 0.5583
[search-T1-KD-t1] Epoch 2/3 | ValAcc: 0.8651 | QryLoss: 0.1163
[search-T1-KD-t1] Epoch 3/3 | ValAcc: 0.8558 | QryLoss: 0.0674
  [search-T1-KD-t1] Best val acc : 0.8651
[I 2026-05-10 16:36:22,854] Trial 1 finished with value: 0.8651162790697674 and parameters: {'inner_lr': 0.0015958237752949746, 'epsilon': 0.15000000000000002, 'lstm_lr': 1.3066739238053272e-05, 'fc_lr': 0.005399484409787433}. Best is trial 1 with value: 0.8651162790697674.
[sear

In [29]:

# extract all best params including the new lr params
T1_KD_INNER_LR = t1_kd_params["inner_lr"]
T1_KD_EPSILON  = t1_kd_params["epsilon"]
T1_KD_LSTM_LR  = t1_kd_params["lstm_lr"]
T1_KD_FC_LR    = t1_kd_params["fc_lr"]

# build episode buffer: T0 exemplars (old) + T1 train (new)
episode_buffer_t1_kd = EpisodeBuffer()
episode_buffer_t1_kd.add_from_loader(exemplar_train_loader, mark_new=False)
episode_buffer_t1_kd.add_from_loader(task1_train_loader,    mark_new=True)
n_way_eff_t1_kd = min(REP_N_WAY, len(episode_buffer_t1_kd.available_classes()))

m_rep_kd_t1 = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)

# final training run with best params
m_rep_kd_t1, _ = train_reptile(
    model=m_rep_kd_t1,
    episode_buffer=episode_buffer_t1_kd,
    val_loader=combined_val_t1_loader,
    device=device,
    teacher=t1_teacher,
    num_old_classes=N0,
    n_way=n_way_eff_t1_kd,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=T1_KD_INNER_LR,
    lstm_lr=T1_KD_LSTM_LR,
    fc_lr=T1_KD_FC_LR,
    inner_steps=T1_INNER_STEPS,
    epsilon=T1_KD_EPSILON,
    episodes=T1_EPISODES,
    epochs=T1_EPOCHS,
    lambda_kd=REP_LAMBDA_KD,
    new_class_bias=T1_NEW_BIAS,
    new_class_ids=set(TASK1_CLASS_IDS),
    new_weight=1.0,
    kd=True,
    tag="Reptile-T1-KD-final",
    trial=None
)

record(
    name="Reptile T1 (KD, best)", model=m_rep_kd_t1, results_dict=results_t1,
    old_loader=task0_test_loader, new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader
)

[Reptile-T1-KD-final] Epoch 1/5 | ValAcc: 0.8558 | QryLoss: 0.0781
[Reptile-T1-KD-final] Epoch 2/5 | ValAcc: 0.8512 | QryLoss: 0.0584
[Reptile-T1-KD-final] Epoch 3/5 | ValAcc: 0.8465 | QryLoss: 0.0433
[Reptile-T1-KD-final] Epoch 4/5 | ValAcc: 0.8372 | QryLoss: 0.0351
[Reptile-T1-KD-final] Epoch 5/5 | ValAcc: 0.8279 | QryLoss: 0.0905
  [Reptile-T1-KD-final] Best val acc : 0.8558
 Reptile T1 (KD, best)                Old:0.9591  New:0.5283  All:0.8571


In [30]:
def print_table(results_dict, title, old_label="Old", new_label="New", extra=""):
    best = max(v["All"] for v in results_dict.values())
    print("\n" + "="*76)
    print(title)
    if extra: print(f"  [{extra}]")
    print("-"*76)
    print(f"{'Variant':<36} {old_label:>10} {new_label:>10} {'All':>10}")
    print("-"*76)
    for name, sc in results_dict.items():
        mark = " ✓" if sc["All"] == best else ""
        print(f"{name:<36} {sc['Old']:>10.4f} {sc['New']:>10.4f} {sc['All']:>10.4f}{mark}")
    print("-"*76)

In [31]:
# print final T1 results across all 6 methods with winner marked
print_table(
    results_dict=results_t1,
    title="TASK 1 RESULTS — Continual Learning after T1",
    old_label="T0 Ret.",
    new_label="T1 Plas.",
    extra="Old=T0 test | New=T1 test | All=T0+T1 test"
)


TASK 1 RESULTS — Continual Learning after T1
  [Old=T0 test | New=T1 test | All=T0+T1 test]
----------------------------------------------------------------------------
Variant                                 T0 Ret.   T1 Plas.        All
----------------------------------------------------------------------------
Rehearsal T1 (no KD, best)               0.9415     0.6226     0.8661
Rehearsal T1 (KD, best)                  0.9415     0.7547     0.8973
Reptile T1 (no KD, best)                 0.9591     0.7170     0.9018 ✓
Reptile T1 (KD, best)                    0.9591     0.5283     0.8571
----------------------------------------------------------------------------


## **TASK 2**

In [32]:
print(f"\n{'='*60}\nOPTUNA SEARCH — Rehearsal T2 (no KD)\n{'='*60}")

# search best lr + lambda_distill for Rehearsal without KD on T2
best_reh_params_t2 = optuna_search_rehearsal(
    task_tag="T2-noKD",
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    exemplar_train_loader=exemplar_train_loader,   # T0 exemplars
    extra_replay_loader=task1_train_loader,        # T1 replay added for T2
    teacher=None,                                  # no KD
    cfg=cfg, device=device, lstm_hidden=256,
    num_classes=N_TOTAL,
    num_old_classes=N0 + N1,                       # T0+T1 old classes
    limit=LIMIT,
    kd=False,
    n_trials=N_TRIALS,
    search_epochs=SEARCH_EPOCHS,
    new_repeat=NEW_REPEAT
)
print("Best Rehearsal (no KD) params for T2:", best_reh_params_t2)

[I 2026-05-10 16:42:15,489] A new study created in memory with name: no-name-062333ae-f831-461e-8298-62646cf63339



OPTUNA SEARCH — Rehearsal T2 (no KD)

OPTUNA REHEARSAL — T2-noKD | 15 trials × 3 epochs


  0%|          | 0/15 [00:00<?, ?it/s]

[Rehearsal no-KD] Epoch 1/3 | Loss: 17.4886 | ValAcc: 0.6679
[Rehearsal no-KD] Epoch 2/3 | Loss: 1.2443 | ValAcc: 0.7687
[Rehearsal no-KD] Epoch 3/3 | Loss: 0.2011 | ValAcc: 0.7388
[I 2026-05-10 16:42:21,129] Trial 0 finished with value: 0.7388059701492538 and parameters: {'lambda_distill': 0.4, 'lstm_lr': 0.0007969454818643932, 'fc_lr': 0.0029106359131330704}. Best is trial 0 with value: 0.7388059701492538.
[Rehearsal no-KD] Epoch 1/3 | Loss: 48.5002 | ValAcc: 0.2015
[Rehearsal no-KD] Epoch 2/3 | Loss: 32.1719 | ValAcc: 0.2463
[Rehearsal no-KD] Epoch 3/3 | Loss: 22.0759 | ValAcc: 0.3396
[I 2026-05-10 16:42:27,167] Trial 1 finished with value: 0.33955223880597013 and parameters: {'lambda_distill': 0.6000000000000001, 'lstm_lr': 2.0513382630874486e-05, 'fc_lr': 0.00020511104188433984}. Best is trial 0 with value: 0.7388059701492538.
[Rehearsal no-KD] Epoch 1/3 | Loss: 21.8796 | ValAcc: 0.7239
[Rehearsal no-KD] Epoch 2/3 | Loss: 2.5872 | ValAcc: 0.7463
[Rehearsal no-KD] Epoch 3/3 | Loss:

In [33]:
# build model + optimizer with best params
m_reh2 = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)
optimizer_reh2 = torch.optim.Adam(
    [
        {"params": m_reh2.lstm.parameters(), "lr": best_reh_params_t2["lstm_lr"]},
        {"params": m_reh2.fc.parameters(),   "lr": best_reh_params_t2["fc_lr"]},
    ],
    weight_decay=1e-4
)

# build replay buffer with T0 exemplars + T1 limited train
replay_buf_t2 = fresh_replay_buffer(
    exemplar_train_loader=combined_exemplar_t2_loader,
    limit=LIMIT
)

# final training run with best params
train_rehearsal(
    model=m_reh2,
    teacher=None,
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    replay_buffer=replay_buf_t2,
    optimizer=optimizer_reh2,
    device=device,
    num_old_classes=N0 + N1,
    lambda_distill=best_reh_params_t2["lambda_distill"],
    epochs=T2_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=False
)

record(
    name="Rehearsal T2 (no KD, best)", model=m_reh2, results_dict=results_t2,
    old_loader=combined_test_t1_loader,   # T0+T1 retention
    new_loader=task2_test_loader,          # T2 plasticity
    all_loader=combined_test_t2_loader
)

[Rehearsal no-KD] Epoch 1/5 | Loss: 14.0331 | ValAcc: 0.6791
[Rehearsal no-KD] Epoch 2/5 | Loss: 0.5347 | ValAcc: 0.7127
[Rehearsal no-KD] Epoch 3/5 | Loss: 0.1143 | ValAcc: 0.7164
[Rehearsal no-KD] Epoch 4/5 | Loss: 0.0441 | ValAcc: 0.7313
[Rehearsal no-KD] Epoch 5/5 | Loss: 0.0322 | ValAcc: 0.7276
 Rehearsal T2 (no KD, best)           Old:0.8214  New:0.4909  All:0.7563


In [34]:
# frozen teacher used for KD loss
teacher_base_t2 = make_teacher(
    fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)
)

# search best lr + lambda_distill for Rehearsal with KD on T2
best_reh_kd_params_t2 = optuna_search_rehearsal(
    task_tag="T2-KD",
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    exemplar_train_loader=exemplar_train_loader,   # T0 exemplars
    extra_replay_loader=task1_train_loader,        # T1 replay added for T2
    teacher=teacher_base_t2,
    cfg=cfg, device=device, lstm_hidden=256,
    num_classes=N_TOTAL,
    num_old_classes=N0 + N1,
    limit=LIMIT,
    kd=True,
    n_trials=N_TRIALS,
    search_epochs=SEARCH_EPOCHS,
    new_repeat=NEW_REPEAT
)
print("Best Rehearsal (KD) params for T2:", best_reh_kd_params_t2)

[I 2026-05-10 16:43:47,801] A new study created in memory with name: no-name-b635f978-2478-4a00-a70c-16ab498935c7



OPTUNA REHEARSAL — T2-KD | 15 trials × 3 epochs


  0%|          | 0/15 [00:00<?, ?it/s]

[Rehearsal KD] Epoch 1/3 | Loss: 19.2093 | ValAcc: 0.7015
[Rehearsal KD] Epoch 2/3 | Loss: 5.6912 | ValAcc: 0.7127
[Rehearsal KD] Epoch 3/3 | Loss: 4.1409 | ValAcc: 0.7127
[I 2026-05-10 16:43:54,431] Trial 0 finished with value: 0.7126865671641791 and parameters: {'lambda_distill': 0.4, 'lstm_lr': 0.0007969454818643932, 'fc_lr': 0.0029106359131330704}. Best is trial 0 with value: 0.7126865671641791.
[Rehearsal KD] Epoch 1/3 | Loss: 50.5999 | ValAcc: 0.2649
[Rehearsal KD] Epoch 2/3 | Loss: 35.4197 | ValAcc: 0.3769
[Rehearsal KD] Epoch 3/3 | Loss: 24.4947 | ValAcc: 0.4478
[I 2026-05-10 16:44:00,054] Trial 1 finished with value: 0.44776119402985076 and parameters: {'lambda_distill': 0.6000000000000001, 'lstm_lr': 2.0513382630874486e-05, 'fc_lr': 0.00020511104188433984}. Best is trial 0 with value: 0.7126865671641791.
[Rehearsal KD] Epoch 1/3 | Loss: 22.1421 | ValAcc: 0.6530
[Rehearsal KD] Epoch 2/3 | Loss: 4.1396 | ValAcc: 0.7201
[Rehearsal KD] Epoch 3/3 | Loss: 2.4063 | ValAcc: 0.7313
[I

In [35]:
# build model + optimizer with best params
m_reh_kd2 = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)
optimizer_reh_kd2 = torch.optim.Adam(
    [
        {"params": m_reh_kd2.lstm.parameters(), "lr": best_reh_kd_params_t2["lstm_lr"]},
        {"params": m_reh_kd2.fc.parameters(),   "lr": best_reh_kd_params_t2["fc_lr"]},
    ],
    weight_decay=1e-4
)

# build replay buffer with T0 exemplars + T1 limited train
replay_buf_t2_kd = fresh_replay_buffer(
    exemplar_train_loader=combined_exemplar_t2_loader,
    limit=LIMIT
)

# final training run with best params
train_rehearsal(
    model=m_reh_kd2,
    teacher=teacher_base_t2,
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    replay_buffer=replay_buf_t2_kd,
    optimizer=optimizer_reh_kd2,
    device=device,
    num_old_classes=N0 + N1,
    lambda_distill=best_reh_kd_params_t2["lambda_distill"],
    epochs=T2_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=True
)

record(
    name="Rehearsal T2 (KD, best)", model=m_reh_kd2, results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader
)

[Rehearsal KD] Epoch 1/5 | Loss: 24.8240 | ValAcc: 0.6642
[Rehearsal KD] Epoch 2/5 | Loss: 10.8313 | ValAcc: 0.7090
[Rehearsal KD] Epoch 3/5 | Loss: 9.3053 | ValAcc: 0.7164
[Rehearsal KD] Epoch 4/5 | Loss: 6.9001 | ValAcc: 0.7127
[Rehearsal KD] Epoch 5/5 | Loss: 6.2438 | ValAcc: 0.7313
 Rehearsal T2 (KD, best)              Old:0.7902  New:0.5455  All:0.7419


In [36]:
# search inner_lr, epsilon, lstm_lr, fc_lr for Reptile without KD on T2
t2_nokd_params = optuna_search_reptile(
    task_tag="T2-noKD",
    val_loader=combined_val_t2_loader,
    new_loaders_for_buffer=[task2_train_loader],
    new_class_ids=set(TASK2_CLASS_IDS),
    search_new_weight=False,
    cfg=cfg, device=device, lstm_hidden=256,
    exemplar_train_loader=combined_exemplar_t2_loader,   # T0 exemplars + T1 train
    num_old_classes=N0 + N1,
    kd_flag=False, teacher=None,
    new_class_bias=T2_NEW_BIAS,
    n_trials=N_TRIALS, search_episodes=SEARCH_EPISODES, search_epochs=SEARCH_EPOCHS,
    rep_n_way=REP_N_WAY, rep_k_support=REP_K_SUPPORT, rep_k_query=REP_K_QUERY,
    rep_lambda_kd=0.0
)
print("Best Reptile (no KD) params for T2:", t2_nokd_params)


OPTUNA REPTILE — T2-noKD (no-KD) | 15 trials | episodes=80 × epochs=3


[I 2026-05-10 16:45:13,630] A new study created in memory with name: no-name-5ce5889f-6544-4834-af0c-ec75b718da4a


  0%|          | 0/15 [00:00<?, ?it/s]

[search-T2-noKD-t0] Epoch 1/3 | ValAcc: 0.7537 | QryLoss: 0.0756
[search-T2-noKD-t0] Epoch 2/3 | ValAcc: 0.7463 | QryLoss: 0.0312
[search-T2-noKD-t0] Epoch 3/3 | ValAcc: 0.7537 | QryLoss: 0.0224
  [search-T2-noKD-t0] Best val acc : 0.7537
[I 2026-05-10 16:45:18,677] Trial 0 finished with value: 0.753731343283582 and parameters: {'inner_lr': 0.0030710573677773722, 'epsilon': 0.5, 'lstm_lr': 0.000291063591313307, 'fc_lr': 0.0015751320499779737}. Best is trial 0 with value: 0.753731343283582.
[search-T2-noKD-t1] Epoch 1/3 | ValAcc: 0.6604 | QryLoss: 0.4466
[search-T2-noKD-t1] Epoch 2/3 | ValAcc: 0.7276 | QryLoss: 0.0775
[search-T2-noKD-t1] Epoch 3/3 | ValAcc: 0.7351 | QryLoss: 0.2902
  [search-T2-noKD-t1] Best val acc : 0.7351
[I 2026-05-10 16:45:24,068] Trial 1 finished with value: 0.7350746268656716 and parameters: {'inner_lr': 0.0015958237752949746, 'epsilon': 0.15000000000000002, 'lstm_lr': 1.3066739238053272e-05, 'fc_lr': 0.005399484409787433}. Best is trial 0 with value: 0.753731343

In [37]:
# extract all best params including lr params
T2_INNER_LR = t2_nokd_params["inner_lr"]
T2_EPSILON  = t2_nokd_params["epsilon"]
T2_LSTM_LR  = t2_nokd_params["lstm_lr"]
T2_FC_LR    = t2_nokd_params["fc_lr"]

# build episode buffer: T0+T1 old classes + T2 new classes
episode_buffer_t2 = EpisodeBuffer()
episode_buffer_t2.add_from_loader(combined_exemplar_t2_loader, mark_new=False)   # T0+T1 old
episode_buffer_t2.add_from_loader(task2_train_loader,          mark_new=True)    # T2 new
n_way_eff_t2 = min(REP_N_WAY, len(episode_buffer_t2.available_classes()))

m_rep_t2 = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)

# final training run with best params
m_rep_t2, _ = train_reptile(
    model=m_rep_t2,
    episode_buffer=episode_buffer_t2,
    val_loader=combined_val_t2_loader,
    device=device,
    teacher=None,
    num_old_classes=N0 + N1,
    n_way=n_way_eff_t2,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=T2_INNER_LR,
    lstm_lr=T2_LSTM_LR,
    fc_lr=T2_FC_LR,
    inner_steps=T2_INNER_STEPS,
    epsilon=T2_EPSILON,
    episodes=T2_EPISODES,
    epochs=T2_EPOCHS,
    lambda_kd=0.0,
    new_class_bias=T2_NEW_BIAS,
    new_class_ids=set(TASK2_CLASS_IDS),
    new_weight=1.0,
    kd=False,
    tag="Reptile-T2-noKD-final",
    trial=None
)

record(
    name="Reptile T2 (no KD, best)", model=m_rep_t2, results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader
)

[Reptile-T2-noKD-final] Epoch 1/5 | ValAcc: 0.7500 | QryLoss: 0.0235
[Reptile-T2-noKD-final] Epoch 2/5 | ValAcc: 0.7388 | QryLoss: 0.0097
[Reptile-T2-noKD-final] Epoch 3/5 | ValAcc: 0.7500 | QryLoss: 0.0091
[Reptile-T2-noKD-final] Epoch 4/5 | ValAcc: 0.7425 | QryLoss: 0.0038
[Reptile-T2-noKD-final] Epoch 5/5 | ValAcc: 0.7425 | QryLoss: 0.0033
  [Reptile-T2-noKD-final] Best val acc : 0.7500
 Reptile T2 (no KD, best)             Old:0.8348  New:0.4727  All:0.7634


In [38]:
# frozen teacher for Reptile KD loss
t2_teacher = make_teacher(
    fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)
)

# search inner_lr, epsilon, lstm_lr, fc_lr for Reptile with KD on T2
t2_kd_params = optuna_search_reptile(
    task_tag="T2-KD",
    val_loader=combined_val_t2_loader,
    new_loaders_for_buffer=[task2_train_loader],
    new_class_ids=set(TASK2_CLASS_IDS),
    search_new_weight=False,
    cfg=cfg, device=device, lstm_hidden=256,
    exemplar_train_loader=combined_exemplar_t2_loader,
    num_old_classes=N0 + N1,
    kd_flag=True, teacher=t2_teacher,
    new_class_bias=T2_NEW_BIAS,
    n_trials=N_TRIALS, search_episodes=SEARCH_EPISODES, search_epochs=SEARCH_EPOCHS,
    rep_n_way=REP_N_WAY, rep_k_support=REP_K_SUPPORT, rep_k_query=REP_K_QUERY,
    rep_lambda_kd=REP_LAMBDA_KD
)
print("Best Reptile (KD) params for T2:", t2_kd_params)


OPTUNA REPTILE — T2-KD (KD) | 15 trials | episodes=80 × epochs=3


[I 2026-05-10 16:46:50,708] A new study created in memory with name: no-name-4f001dab-7603-4712-8bea-4b54926d1f9b


  0%|          | 0/15 [00:00<?, ?it/s]

[search-T2-KD-t0] Epoch 1/3 | ValAcc: 0.7500 | QryLoss: 0.0589
[search-T2-KD-t0] Epoch 2/3 | ValAcc: 0.7425 | QryLoss: 0.0103
[search-T2-KD-t0] Epoch 3/3 | ValAcc: 0.7463 | QryLoss: 0.0617
  [search-T2-KD-t0] Best val acc : 0.7500
[I 2026-05-10 16:46:57,852] Trial 0 finished with value: 0.75 and parameters: {'inner_lr': 0.0030710573677773722, 'epsilon': 0.5, 'lstm_lr': 0.000291063591313307, 'fc_lr': 0.0015751320499779737}. Best is trial 0 with value: 0.75.
[search-T2-KD-t1] Epoch 1/3 | ValAcc: 0.6866 | QryLoss: 0.8585
[search-T2-KD-t1] Epoch 2/3 | ValAcc: 0.7127 | QryLoss: 0.5240
[search-T2-KD-t1] Epoch 3/3 | ValAcc: 0.7537 | QryLoss: 0.1135
  [search-T2-KD-t1] Best val acc : 0.7537
[I 2026-05-10 16:47:05,114] Trial 1 finished with value: 0.753731343283582 and parameters: {'inner_lr': 0.0015958237752949746, 'epsilon': 0.15000000000000002, 'lstm_lr': 1.3066739238053272e-05, 'fc_lr': 0.005399484409787433}. Best is trial 1 with value: 0.753731343283582.
[search-T2-KD-t2] Epoch 1/3 | ValAc

In [39]:
# extract all best params including lr params
T2_KD_INNER_LR = t2_kd_params["inner_lr"]
T2_KD_EPSILON  = t2_kd_params["epsilon"]
T2_KD_LSTM_LR  = t2_kd_params["lstm_lr"]
T2_KD_FC_LR    = t2_kd_params["fc_lr"]

# build episode buffer: T0+T1 old classes + T2 new classes
episode_buffer_t2_kd = EpisodeBuffer()
episode_buffer_t2_kd.add_from_loader(combined_exemplar_t2_loader, mark_new=False)
episode_buffer_t2_kd.add_from_loader(task2_train_loader,          mark_new=True)
n_way_eff_t2_kd = min(REP_N_WAY, len(episode_buffer_t2_kd.available_classes()))

m_rep_kd_t2 = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)

# final training run with best params
m_rep_kd_t2, _ = train_reptile(
    model=m_rep_kd_t2,
    episode_buffer=episode_buffer_t2_kd,
    val_loader=combined_val_t2_loader,
    device=device,
    teacher=t2_teacher,
    num_old_classes=N0 + N1,
    n_way=n_way_eff_t2_kd,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=T2_KD_INNER_LR,
    lstm_lr=T2_KD_LSTM_LR,
    fc_lr=T2_KD_FC_LR,
    inner_steps=T2_INNER_STEPS,
    epsilon=T2_KD_EPSILON,
    episodes=T2_EPISODES,
    epochs=T2_EPOCHS,
    lambda_kd=REP_LAMBDA_KD,
    new_class_bias=T2_NEW_BIAS,
    new_class_ids=set(TASK2_CLASS_IDS),
    new_weight=1.0,
    kd=True,
    tag="Reptile-T2-KD-final",
    trial=None
)

record(
    name="Reptile T2 (KD, best)", model=m_rep_kd_t2, results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader
)

[Reptile-T2-KD-final] Epoch 1/5 | ValAcc: 0.7575 | QryLoss: 0.0529
[Reptile-T2-KD-final] Epoch 2/5 | ValAcc: 0.7500 | QryLoss: 0.0261
[Reptile-T2-KD-final] Epoch 3/5 | ValAcc: 0.7351 | QryLoss: 0.0584
[Reptile-T2-KD-final] Epoch 4/5 | ValAcc: 0.7388 | QryLoss: 0.0199
[Reptile-T2-KD-final] Epoch 5/5 | ValAcc: 0.7388 | QryLoss: 0.0133
  [Reptile-T2-KD-final] Best val acc : 0.7575
 Reptile T2 (KD, best)                Old:0.8482  New:0.5091  All:0.7814


In [40]:
print_table(
    results_dict=results_t2,
    title="TASK 2 RESULTS — Continual Learning after T2",
    old_label="T0+T1 Ret.",
    new_label="T2 Plas.",
    extra=f"Old=T0+T1 test | New=T2 test | All=T0+T1+T2 test"
)


TASK 2 RESULTS — Continual Learning after T2
  [Old=T0+T1 test | New=T2 test | All=T0+T1+T2 test]
----------------------------------------------------------------------------
Variant                              T0+T1 Ret.   T2 Plas.        All
----------------------------------------------------------------------------
Rehearsal T2 (no KD, best)               0.8214     0.4909     0.7563
Rehearsal T2 (KD, best)                  0.7902     0.5455     0.7419
Reptile T2 (no KD, best)                 0.8348     0.4727     0.7634
Reptile T2 (KD, best)                    0.8482     0.5091     0.7814 ✓
----------------------------------------------------------------------------
